# Sequence: Colab Data Collection & Training

This notebook enables data collection and model training for the Sequence framework using Google Colab's GPUs and Google Drive for data storage.

**Key Features:**
- 🔗 Mount Google Drive for persistent data storage
- 📊 Collect HistData, GDELT sentiment, and fundamental economic data
- 🎯 Prepare datasets with intrinsic time transformation
- 🚀 Train models on Colab GPUs (T4/P100/V100/A100)
- 💾 Store all data and checkpoints on Google Drive (no local storage limits)

**Storage Strategy:**
- Raw data: `MyDrive/Sequence/data/raw/`
- Prepared datasets: `MyDrive/Sequence/data/prepared/`
- Model checkpoints: `MyDrive/Sequence/models/`
- Logs: `MyDrive/Sequence/logs/`

---

## 1. Environment Setup

### Step 1.1: Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define base paths
DRIVE_ROOT = Path('/content/drive/MyDrive/Sequence')
REPO_ROOT = Path('/content/Sequence')

# Create directory structure on Google Drive
directories = [
    DRIVE_ROOT / 'data' / 'raw' / 'histdata',
    DRIVE_ROOT / 'data' / 'raw' / 'gdelt',
    DRIVE_ROOT / 'data' / 'raw' / 'fundamentals',
    DRIVE_ROOT / 'data' / 'prepared',
    DRIVE_ROOT / 'models',
    DRIVE_ROOT / 'logs',
    DRIVE_ROOT / 'config',
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"✓ {directory}")

print(f"\n📁 Drive root: {DRIVE_ROOT}")
print(f"💾 Available space: {os.popen('df -h /content/drive | tail -1').read().split()[3]}")

### Step 1.2: Clone/Update Repository

In [ ]:
import subprocess
import sys

# Clone or update repository
if REPO_ROOT.exists():
    print("📦 Updating existing repository...")
    !cd {REPO_ROOT} && git pull origin main
else:
    print("📦 Cloning repository...")
    !git clone https://github.com/crichalchemist/Sequence.git {REPO_ROOT}

# Add to Python path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'run') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'run'))

print(f"\n✓ Repository ready: {REPO_ROOT}")
print(f"✓ Current branch: {subprocess.check_output(['git', '-C', str(REPO_ROOT), 'branch', '--show-current']).decode().strip()}")
print(f"✓ Latest commit: {subprocess.check_output(['git', '-C', str(REPO_ROOT), 'log', '-1', '--oneline']).decode().strip()}")

### Step 1.3: Install Dependencies

In [ ]:
# Install core dependencies
print("📦 Installing core dependencies...")
!pip install -q -r {REPO_ROOT}/requirements.txt

# Install fundamental data sources
print("\n📦 Installing fundamental data sources...")
!pip install -q -e {REPO_ROOT}/new_data_sources/comtradeapicall
!pip install -q -e {REPO_ROOT}/new_data_sources/FRB

# Install TimesFM (Google's foundation model)
if (REPO_ROOT / 'models' / 'timesFM').exists():
    print("\n📦 Installing TimesFM...")
    !pip install -q -e {REPO_ROOT}/models/timesFM

print("\n✓ All dependencies installed!")

# Verify key packages
import torch
import pandas as pd
import transformers

print(f"\n📊 Package versions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Step 1.4: Configure API Keys (Secure Storage)

**IMPORTANT:** Store API keys securely on Google Drive, not in the notebook!

In [ ]:
from google.colab import userdata
import json

# Option 1: Store keys in Colab Secrets (Recommended)
# Go to: 🔑 (key icon) in left sidebar → Add secrets
# Add: FRED_API_KEY, COMTRADE_API_KEY (optional)

# Option 2: Load from Drive config file
config_file = DRIVE_ROOT / 'config' / 'api_keys.json'

def get_api_keys():
    """Load API keys from Colab Secrets or Drive config."""
    keys = {}
    
    # Try Colab Secrets first
    try:
        keys['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
        print("✓ Loaded FRED_API_KEY from Colab Secrets")
    except:
        pass
    
    try:
        keys['COMTRADE_API_KEY'] = userdata.get('COMTRADE_API_KEY')
        print("✓ Loaded COMTRADE_API_KEY from Colab Secrets")
    except:
        pass
    
    # Fall back to Drive config file
    if not keys and config_file.exists():
        with open(config_file) as f:
            keys = json.load(f)
        print(f"✓ Loaded {len(keys)} keys from {config_file}")
    
    # Set environment variables
    for key, value in keys.items():
        os.environ[key] = value
    
    return keys

# Load keys
api_keys = get_api_keys()

if 'FRED_API_KEY' not in api_keys:
    print("\n⚠️  FRED_API_KEY not found!")
    print("   Get free key: https://fred.stlouisfed.org/docs/api/api_key.html")
    print("   Then add to Colab Secrets or create config file:")
    print(f"   {config_file}")
    print('   Format: {"FRED_API_KEY": "your_key_here"}')
else:
    print(f"\n✓ API keys configured: {', '.join(api_keys.keys())}")

# Helper to create config file (run once, then delete this cell)
def save_api_keys(fred_key, comtrade_key=None):
    """Save API keys to Drive config (one-time setup)."""
    keys = {'FRED_API_KEY': fred_key}
    if comtrade_key:
        keys['COMTRADE_API_KEY'] = comtrade_key
    
    with open(config_file, 'w') as f:
        json.dump(keys, f, indent=2)
    print(f"✓ Saved keys to {config_file}")

# Uncomment and run once to save keys:
# save_api_keys(fred_key='YOUR_FRED_API_KEY_HERE', comtrade_key='YOUR_COMTRADE_KEY')

## 2. Data Collection

### Step 2.1: Configure Collection Parameters

In [ ]:
# Data collection configuration
COLLECTION_CONFIG = {
    # Currency pairs to collect
    'pairs': ['gbpusd', 'eurusd'],  # Add more: 'usdjpy', 'audusd', etc.
    
    # Date range
    'start_date': '2020-01-01',
    'end_date': '2023-12-31',
    
    # Data sources
    'collect_histdata': True,      # Price data
    'collect_gdelt': False,         # Sentiment (slow, optional)
    'collect_fundamentals': True,  # Economic indicators
    
    # Preparation settings
    't_in': 120,                   # Input sequence length
    't_out': 10,                   # Prediction horizon
    'use_intrinsic_time': True,    # Directional-change bars
    'dc_threshold': 0.0005,        # 5 pips for major FX pairs
    'include_sentiment': False,    # Include GDELT features
    'task_type': 'classification', # or 'regression'
}

print("📋 Data Collection Configuration:")
print(f"  Pairs: {', '.join(COLLECTION_CONFIG['pairs'])}")
print(f"  Date range: {COLLECTION_CONFIG['start_date']} to {COLLECTION_CONFIG['end_date']}")
print(f"  Intrinsic time: {COLLECTION_CONFIG['use_intrinsic_time']}")
print(f"  Data sources: HistData={COLLECTION_CONFIG['collect_histdata']}, "
      f"GDELT={COLLECTION_CONFIG['collect_gdelt']}, "
      f"Fundamentals={COLLECTION_CONFIG['collect_fundamentals']}")

### Step 2.2: Collect HistData (Price Data)

Downloads tick-by-tick FX data from HistData and stores on Google Drive.

In [ ]:
if COLLECTION_CONFIG['collect_histdata']:
    import histdata
    from datetime import datetime
    
    # Set output directory to Google Drive
    histdata_output = DRIVE_ROOT / 'data' / 'raw' / 'histdata'
    
    print("📊 Collecting HistData...")
    print(f"   Output: {histdata_output}\n")
    
    for pair in COLLECTION_CONFIG['pairs']:
        print(f"\n{'='*60}")
        print(f"Processing: {pair.upper()}")
        print(f"{'='*60}")
        
        pair_dir = histdata_output / pair
        pair_dir.mkdir(exist_ok=True)
        
        # Parse date range
        start = datetime.strptime(COLLECTION_CONFIG['start_date'], '%Y-%m-%d')
        end = datetime.strptime(COLLECTION_CONFIG['end_date'], '%Y-%m-%d')
        
        # Download data
        # Note: histdata library downloads monthly ZIP files
        # This can be slow for large date ranges
        try:
            !cd {pair_dir} && python -m histdata.download \
                --pair {pair} \
                --year {start.year}-{end.year} \
                --format tick-data-quotes
            
            # Check collected files
            csv_files = list(pair_dir.glob('*.csv'))
            print(f"\n✓ Collected {len(csv_files)} files for {pair.upper()}")
            
            # Show file sizes
            total_size = sum(f.stat().st_size for f in csv_files)
            print(f"  Total size: {total_size / 1e9:.2f} GB")
            
        except Exception as e:
            print(f"\n❌ Error collecting {pair}: {e}")
            print("   Try manual download from: http://www.histdata.com/download-free-forex-data/")
    
    print("\n✓ HistData collection complete!")
else:
    print("⏭️  Skipping HistData collection (disabled in config)")

### Step 2.3: Collect Fundamental Data

Downloads economic indicators (FRED), trade data (Comtrade), and monetary policy shocks (ECB).

In [ ]:
if COLLECTION_CONFIG['collect_fundamentals']:
    os.chdir(REPO_ROOT)
    
    from data.extended_data_collection import collect_all_forex_fundamentals
    
    fundamentals_output = DRIVE_ROOT / 'data' / 'raw' / 'fundamentals'
    
    print("📊 Collecting fundamental data...")
    print(f"   Output: {fundamentals_output}\n")
    
    if 'FRED_API_KEY' not in os.environ:
        print("❌ FRED_API_KEY not set! Skipping fundamental data collection.")
        print("   Configure API key in Step 1.4")
    else:
        for pair in COLLECTION_CONFIG['pairs']:
            print(f"\n{'='*60}")
            print(f"Processing: {pair.upper()}")
            print(f"{'='*60}")
            
            try:
                # Collect all fundamental data sources
                data = collect_all_forex_fundamentals(
                    currency_pair=pair.upper(),
                    start_date=COLLECTION_CONFIG['start_date'],
                    end_date=COLLECTION_CONFIG['end_date'],
                    fred_api_key=os.environ.get('FRED_API_KEY'),
                    comtrade_api_key=os.environ.get('COMTRADE_API_KEY'),  # Optional
                )
                
                # Save each data source as CSV (primary storage format)
                pair_dir = fundamentals_output / pair
                pair_dir.mkdir(exist_ok=True)
                
                for source_name, df in data.items():
                    if df is not None and len(df) > 0:
                        output_file = pair_dir / f"{source_name}.csv"
                        df.to_csv(output_file, index=True)
                        print(f"  ✓ {source_name}: {len(df)} records → {output_file.name}")
                    else:
                        print(f"  ⚠️  {source_name}: No data")
                
                print(f"\n  ✓ Saved fundamental data for {pair.upper()}")
                
            except Exception as e:
                print(f"\n❌ Error collecting fundamentals for {pair}: {e}")
                import traceback
                traceback.print_exc()
        
        print("\n✓ Fundamental data collection complete!")
else:
    print("⏭️  Skipping fundamental data collection (disabled in config)")

### Step 2.4: Collect GDELT Sentiment (Optional)

**Warning:** GDELT collection is slow and requires significant storage. Only enable if you need sentiment features.

In [ ]:
if COLLECTION_CONFIG['collect_gdelt']:
    print("📰 Collecting GDELT sentiment data...")
    print("⚠️  This may take several hours for multi-year date ranges!\n")
    
    gdelt_output = DRIVE_ROOT / 'data' / 'raw' / 'gdelt'
    
    for pair in COLLECTION_CONFIG['pairs']:
        print(f"\nProcessing GDELT for {pair.upper()}...")
        
        # Extract currencies for keyword filtering
        base = pair[:3].upper()
        quote = pair[3:].upper()
        
        # Run GDELT downloader
        !cd {REPO_ROOT}/data/gdelt && python consolidated_downloader.py \
            --start-date {COLLECTION_CONFIG['start_date']} \
            --end-date {COLLECTION_CONFIG['end_date']} \
            --keywords "{base},{quote},forex,currency,central bank" \
            --output-dir {gdelt_output / pair}
    
    print("\n✓ GDELT collection complete!")
else:
    print("⏭️  Skipping GDELT collection (disabled in config)")
    print("   Enable with: COLLECTION_CONFIG['collect_gdelt'] = True")

## 3. Data Preparation

### Step 3.1: Prepare Datasets with Feature Engineering

In [ ]:
os.chdir(REPO_ROOT)

prepared_output = DRIVE_ROOT / 'data' / 'prepared'

print("🔧 Preparing datasets with feature engineering...")
print(f"   Output: {prepared_output}\n")

for pair in COLLECTION_CONFIG['pairs']:
    print(f"\n{'='*60}")
    print(f"Preparing: {pair.upper()}")
    print(f"{'='*60}")
    
    # Build command
    cmd = [
        'python', f'{REPO_ROOT}/data/prepare_dataset.py',
        '--pairs', pair,
        '--t-in', str(COLLECTION_CONFIG['t_in']),
        '--t-out', str(COLLECTION_CONFIG['t_out']),
        '--task-type', COLLECTION_CONFIG['task_type'],
        '--output-dir', str(prepared_output),
    ]
    
    # Add optional flags
    if COLLECTION_CONFIG['use_intrinsic_time']:
        cmd.extend(['--intrinsic-time'])
        cmd.extend(['--dc-threshold-up', str(COLLECTION_CONFIG['dc_threshold'])])
    
    if COLLECTION_CONFIG['include_sentiment']:
        cmd.extend(['--include-sentiment'])
    
    # Set input path to Drive
    histdata_path = DRIVE_ROOT / 'data' / 'raw' / 'histdata' / pair
    if histdata_path.exists():
        cmd.extend(['--input-dir', str(histdata_path)])
    
    # Run preparation
    print(f"\nCommand: {' '.join(cmd)}\n")
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        
        if result.returncode == 0:
            print("✓ Preparation successful!")
            
            # Check output
            prepared_file = prepared_output / pair / f"{pair}_prepared.csv"
            if prepared_file.exists():
                import pandas as pd
                df = pd.read_csv(prepared_file, nrows=5)
                print(f"\n  File: {prepared_file}")
                print(f"  Columns: {len(df.columns)}")
                print(f"  Features: {', '.join(df.columns[:10])}...")
                print(f"  Size: {prepared_file.stat().st_size / 1e6:.1f} MB")
        else:
            print(f"❌ Preparation failed!")
            print(result.stderr)
            
    except subprocess.TimeoutExpired:
        print("❌ Preparation timed out (>1 hour)")
    except Exception as e:
        print(f"❌ Error: {e}")

print("\n✓ All data preparation complete!")

### Step 3.2: Validate Prepared Data

In [ ]:
import pandas as pd

print("🔍 Validating prepared datasets...\n")

for pair in COLLECTION_CONFIG['pairs']:
    prepared_file = prepared_output / pair / f"{pair}_prepared.csv"
    
    if not prepared_file.exists():
        print(f"❌ {pair.upper()}: File not found")
        continue
    
    try:
        df = pd.read_csv(prepared_file)
        
        print(f"\n{'='*60}")
        print(f"{pair.upper()}")
        print(f"{'='*60}")
        print(f"  Rows: {len(df):,}")
        print(f"  Columns: {len(df.columns)}")
        print(f"  Size: {prepared_file.stat().st_size / 1e6:.1f} MB")
        print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
        
        # Check for missing values
        missing = df.isnull().sum()
        if missing.sum() > 0:
            print(f"  ⚠️  Missing values: {missing[missing > 0].to_dict()}")
        else:
            print(f"  ✓ No missing values")
        
        # Check for infinite values
        inf_cols = [col for col in df.select_dtypes(include=['float64']).columns 
                    if df[col].isin([float('inf'), float('-inf')]).any()]
        if inf_cols:
            print(f"  ⚠️  Infinite values in: {', '.join(inf_cols)}")
        else:
            print(f"  ✓ No infinite values")
        
        # Sample data
        print(f"\n  First 3 rows:")
        print(df.head(3).to_string(max_cols=8))
        
        print(f"\n  ✓ {pair.upper()} validation passed!")
        
    except Exception as e:
        print(f"❌ {pair.upper()}: Validation error - {e}")

print("\n✓ Validation complete!")

## 4. Model Training

### Step 4.1: Configure Training Parameters

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'pair': 'gbpusd',  # Select from prepared pairs
    'epochs': 50,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'checkpoint_dir': str(DRIVE_ROOT / 'models'),
    'log_dir': str(DRIVE_ROOT / 'logs'),
    
    # Model architecture
    'lstm_hidden_size': 128,
    'attention_dim': 64,
    'num_attention_heads': 4,
    
    # Training type
    'training_type': 'supervised',  # 'supervised', 'multitask', or 'rl'
}

print("🚀 Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

# Check GPU
if torch.cuda.is_available():
    print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  No GPU detected! Training will be slow on CPU.")
    print("   Runtime → Change runtime type → GPU (T4)")

### Step 4.2: Supervised Training

Train CNN-LSTM-Attention model for price prediction.

In [ ]:
if TRAINING_CONFIG['training_type'] == 'supervised':
    os.chdir(REPO_ROOT)
    
    # Get data path from Drive
    data_path = prepared_output / TRAINING_CONFIG['pair'] / f"{TRAINING_CONFIG['pair']}_prepared.csv"
    
    if not data_path.exists():
        print(f"❌ Data not found: {data_path}")
        print("   Run Step 3.1 to prepare data first")
    else:
        print(f"🚀 Starting supervised training...")
        print(f"   Data: {data_path}")
        print(f"   Checkpoints: {TRAINING_CONFIG['checkpoint_dir']}\n")
        
        # Run training script
        !python {REPO_ROOT}/train/run_training.py \
            --pairs {TRAINING_CONFIG['pair']} \
            --epochs {TRAINING_CONFIG['epochs']} \
            --learning-rate {TRAINING_CONFIG['learning_rate']} \
            --batch-size {TRAINING_CONFIG['batch_size']} \
            --device {TRAINING_CONFIG['device']} \
            --checkpoint-dir {TRAINING_CONFIG['checkpoint_dir']} \
            --data-path {data_path}
        
        print("\n✓ Training complete!")
        
        # Check saved checkpoints
        checkpoint_dir = Path(TRAINING_CONFIG['checkpoint_dir'])
        checkpoints = list(checkpoint_dir.glob('*.pt'))
        print(f"\n📦 Saved checkpoints: {len(checkpoints)}")
        for ckpt in checkpoints[:3]:
            print(f"  {ckpt.name} ({ckpt.stat().st_size / 1e6:.1f} MB)")
else:
    print("⏭️  Skipping supervised training")
    print(f"   Current mode: {TRAINING_CONFIG['training_type']}")

### Step 4.3: Multi-Task Training (Optional)

Train model to jointly predict price, volatility, and market regime.

In [ ]:
if TRAINING_CONFIG['training_type'] == 'multitask':
    os.chdir(REPO_ROOT)
    
    data_path = prepared_output / TRAINING_CONFIG['pair'] / f"{TRAINING_CONFIG['pair']}_prepared.csv"
    
    if not data_path.exists():
        print(f"❌ Data not found: {data_path}")
    else:
        print(f"🚀 Starting multi-task training...\n")
        
        !python {REPO_ROOT}/train/run_training_multitask.py \
            --pairs {TRAINING_CONFIG['pair']} \
            --epochs {TRAINING_CONFIG['epochs']} \
            --batch-size {TRAINING_CONFIG['batch_size']} \
            --device {TRAINING_CONFIG['device']} \
            --checkpoint-dir {TRAINING_CONFIG['checkpoint_dir']} \
            --data-path {data_path}
        
        print("\n✓ Multi-task training complete!")
else:
    print("⏭️  Skipping multi-task training")

### Step 4.4: Reinforcement Learning Training (Optional)

Train A3C agent for optimal execution policy.

In [ ]:
if TRAINING_CONFIG['training_type'] == 'rl':
    os.chdir(REPO_ROOT)
    
    RL_CONFIG = {
        'env_mode': 'backtesting',  # or 'simulated'
        'num_workers': 8,
        'total_steps': 1_000_000,
    }
    
    data_path = prepared_output / TRAINING_CONFIG['pair'] / f"{TRAINING_CONFIG['pair']}_prepared.csv"
    
    if not data_path.exists():
        print(f"❌ Data not found: {data_path}")
    else:
        print(f"🚀 Starting RL training (A3C)...")
        print(f"   Environment: {RL_CONFIG['env_mode']}")
        print(f"   Workers: {RL_CONFIG['num_workers']}\n")
        
        !python {REPO_ROOT}/rl/run_a3c_training.py \
            --pair {TRAINING_CONFIG['pair']} \
            --env-mode {RL_CONFIG['env_mode']} \
            --historical-data {data_path} \
            --num-workers {RL_CONFIG['num_workers']} \
            --total-steps {RL_CONFIG['total_steps']} \
            --device {TRAINING_CONFIG['device']} \
            --checkpoint-dir {TRAINING_CONFIG['checkpoint_dir']}
        
        print("\n✓ RL training complete!")
else:
    print("⏭️  Skipping RL training")

## 5. Monitoring & Utilities

### Step 5.1: Check Storage Usage

In [ ]:
import subprocess

def get_dir_size(path):
    """Get directory size in GB."""
    try:
        result = subprocess.check_output(['du', '-sh', str(path)]).decode()
        return result.split()[0]
    except:
        return "N/A"

print("💾 Storage Usage on Google Drive:\n")
print(f"{'='*60}")

directories = [
    ('Raw HistData', DRIVE_ROOT / 'data' / 'raw' / 'histdata'),
    ('Raw GDELT', DRIVE_ROOT / 'data' / 'raw' / 'gdelt'),
    ('Raw Fundamentals', DRIVE_ROOT / 'data' / 'raw' / 'fundamentals'),
    ('Prepared Datasets', DRIVE_ROOT / 'data' / 'prepared'),
    ('Model Checkpoints', DRIVE_ROOT / 'models'),
    ('Logs', DRIVE_ROOT / 'logs'),
    ('', None),  # Separator
    ('TOTAL', DRIVE_ROOT),
]

for name, path in directories:
    if path is None:
        print(f"{'─'*60}")
        continue
    
    size = get_dir_size(path) if path.exists() else "0"
    exists = "✓" if path.exists() else "✗"
    print(f"{exists} {name:.<40} {size:>10}")

print(f"{'='*60}")

# Drive quota
df_output = subprocess.check_output(['df', '-h', '/content/drive']).decode()
print(f"\nGoogle Drive Quota:")
print(df_output)

### Step 5.2: Monitor GPU Usage

In [ ]:
if torch.cuda.is_available():
    print("🎮 GPU Status:\n")
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")
    
    print("\n📊 nvidia-smi output:\n")
    !nvidia-smi
else:
    print("⚠️  No GPU available")
    print("   Change runtime: Runtime → Change runtime type → GPU")

### Step 5.3: Backup Critical Files

In [ ]:
from datetime import datetime
import shutil

# Create timestamped backup
backup_dir = DRIVE_ROOT / 'backups' / datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 Creating backup: {backup_dir}\n")

# Backup model checkpoints
models_dir = DRIVE_ROOT / 'models'
if models_dir.exists():
    checkpoints = list(models_dir.glob('*.pt'))
    for ckpt in checkpoints:
        shutil.copy2(ckpt, backup_dir / ckpt.name)
        print(f"✓ Backed up: {ckpt.name}")

# Backup configuration
config_dir = DRIVE_ROOT / 'config'
if config_dir.exists():
    for config_file in config_dir.glob('*.json'):
        shutil.copy2(config_file, backup_dir / config_file.name)
        print(f"✓ Backed up: {config_file.name}")

print(f"\n✓ Backup complete: {backup_dir}")

## 6. Summary & Next Steps

### Quick Status Check

In [ ]:
print("📋 Sequence Framework Status\n")
print(f"{'='*60}")

# Environment
print(f"\n🔧 Environment:")
print(f"  Repository: {REPO_ROOT}")
print(f"  Drive root: {DRIVE_ROOT}")
print(f"  GPU: {'✓ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '✗ CPU only'}")

# Data
print(f"\n📊 Data:")
prepared_pairs = []
if (prepared_output).exists():
    prepared_pairs = [p.name for p in prepared_output.iterdir() if p.is_dir()]
print(f"  Prepared pairs: {', '.join(prepared_pairs) if prepared_pairs else 'None'}")

# Models
print(f"\n🤖 Models:")
models_dir = DRIVE_ROOT / 'models'
if models_dir.exists():
    checkpoints = list(models_dir.glob('*.pt'))
    print(f"  Checkpoints: {len(checkpoints)}")
    for ckpt in checkpoints[:3]:
        print(f"    - {ckpt.name}")
else:
    print(f"  Checkpoints: None")

# Storage
print(f"\n💾 Storage:")
total_size = get_dir_size(DRIVE_ROOT)
print(f"  Total used: {total_size}")

print(f"\n{'='*60}")
print(f"\n✨ Next Steps:")
print(f"  1. Collect more data: Modify COLLECTION_CONFIG and run Step 2")
print(f"  2. Train models: Configure TRAINING_CONFIG and run Step 4")
print(f"  3. Evaluate: Run evaluation scripts in eval/ directory")
print(f"  4. Backup: Regularly backup models to Drive (Step 5.3)")
print(f"\n📖 Documentation: {REPO_ROOT}/CLAUDE.md")
print(f"🐛 Issues: https://github.com/crichalchemist/Sequence/issues")

---

## 💡 Tips & Troubleshooting

### Storage Management
- Google Drive free tier: 15 GB
- HistData tick data: ~500 MB per pair per year
- Prepared datasets: ~100-200 MB per pair
- Model checkpoints: ~50-100 MB each
- **Tip:** Use `COLLECTION_CONFIG['pairs']` to limit pairs if storage is tight

### GPU Quotas
- Free tier: Limited GPU hours per week
- Colab Pro: More GPU time + faster GPUs (V100/A100)
- **Tip:** Train during off-peak hours for better GPU availability

### Data Collection
- HistData: Can be slow; consider manual download for large ranges
- GDELT: Very slow; only enable if you need sentiment features
- Fundamentals: Fast with API keys; FRED key is free and recommended

### Training
- Start with small epochs (10-20) to verify setup
- Monitor GPU memory in Step 5.2
- Use smaller batch sizes if OOM errors occur
- Checkpoints auto-save to Drive (no data loss)

### Common Issues
1. **"No space left"**: Data on local disk, not Drive → Check paths in configs
2. **"CUDA out of memory"**: Reduce batch size or use CPU
3. **"FRED_API_KEY not found"**: Set up API key in Step 1.4
4. **"Data not found"**: Run preparation (Step 3) before training (Step 4)

---

**Created:** 2026-01-12  
**Framework:** Sequence v1.0  
**Colab Runtime:** Python 3.10+ with GPU support